## COMP90024 Team 2



## Scenario 7: Real-time Melbourne Mood Monitor

**COMP90024 — Cluster and Cloud Computing — Assignment 2**

**Team \<2\>**

---

### Goal

Demonstrate live data ingestion via Fission + ElasticSearch, and compare today's Melbourne weather with observed social media sentiment in real time.

### Architecture

Fission timer triggers → harvest Reddit / Mastodon / BlueSky / BOM → store in ElasticSearch → Fission REST API → this notebook

### Data Pipeline

All data is fetched through the single Fission REST API endpoint:

```
GET http://localhost:9090/api/query
```

Run before executing:

```bash
kubectl port-forward -n fission service/router 9090:80
```

### Relationship to Exploratory Analysis

The exploratory notebook analysed the full historical corpus across Sydney, Melbourne and Brisbane (Scenarios 1–6). This notebook focuses on **Melbourne only**, using **real-time data** to compare with historical observations on a live system.

Set `ANALYTICS_API_URL` to use a different endpoint. Analysis requires a populated
cleaned-post index with city/date, sentiment and matched weather fields. Empty
required windows stop with an explanatory message. API connectivity alone does
not establish harvester health or a causal weather effect.


---

## 6.0 Setup & Connection

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from wordcloud import WordCloud, STOPWORDS
from scipy.stats import pearsonr


# ── Unified plot styling (matches exploratory notebook) ──────
sns.set_theme(style='whitegrid', context='notebook',
              palette='muted', font_scale=1.0)

rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.8,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.titlepad': 12,
    'axes.labelsize': 11,
    'axes.labelcolor': '#333333',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.color': '#555555',
    'ytick.color': '#555555',
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.frameon': False,
    'legend.fontsize': 10,
    'grid.color': '#E5E5E5',
    'grid.linewidth': 0.6,
    'figure.titlesize': 15,
    'figure.titleweight': 'bold',
    'savefig.dpi': 110,
    'figure.dpi': 100,
})

# ── Connection ───────────────────────────────────────────────
import os
API_BASE = os.environ.get("ANALYTICS_API_URL", "http://localhost:9090/api/query")

def api_get(**params):
    """Call the Fission REST API."""
    response = requests.get(API_BASE, params=params, timeout=120)
    response.raise_for_status()
    return response.json()

# ── Palette (consistent with exploratory) ────────────────────
CITY_COLORS = {
    'sydney':    '#E07A1F',
    'melbourne': '#1F6FB4',
    'brisbane':  '#2E8B57',
}
PLATFORM_COLORS = {
    'reddit':   '#FF4500',
    'mastodon': '#6364FF',
    'bluesky':  '#0085FF',
}
SENTIMENT_COLORS = {
    'positive': '#4CAF50',
    'neutral':  '#9E9E9E',
    'negative': '#E53935',
}

TODAY = datetime.now(ZoneInfo("Australia/Melbourne")).strftime('%Y-%m-%d')
SEVEN_DAYS_AGO = (datetime.now(ZoneInfo("Australia/Melbourne")) - timedelta(days=7)).strftime('%Y-%m-%d')

print(f'API endpoint : {API_BASE}')
print(f'Today        : {TODAY}')
print(f"Notebook run : {datetime.now(ZoneInfo('Australia/Melbourne')).strftime('%Y-%m-%d %H:%M:%S')}")

def require_frame(frame, columns, context):
    """Stop with actionable input requirements before constructing a plot."""
    missing = sorted(set(columns) - set(frame.columns))
    if frame.empty or missing:
        raise RuntimeError(
            f'{context}: no usable records or missing fields {missing}. '
            'Check API connectivity, the cleaned index, weather joins and date filters.'
        )
    return frame

---

## 6.1 System Status Dashboard

Verify that the Fission API and ElasticSearch are running, and data is flowing.

In [ ]:
# ── Health check ─────────────────────────────────────────────
health = api_get(mode="health")

# ── Summary (full corpus) ───────────────────────────────────
summary = api_get(mode="summary")
if not sum(summary.get('by_city', {}).values()):
    raise RuntimeError('The cleaned-post index is empty. Run ingestion and cleaning before this analysis.')

# ── Today's posts ───────────────────────────────────────────
today_payload = api_get(mode="posts", city="melbourne",
                        size=1000, **{"from": TODAY, "to": TODAY})
today_posts = pd.DataFrame(today_payload.get("rows", []))
today_count = len(today_posts)

if today_count > 0:
    today_posts['sentiment'] = pd.to_numeric(today_posts['sentiment'], errors='coerce')
    for col in ['tmax', 'prcp', 'tavg']:
        if col in today_posts.columns:
            today_posts[col] = pd.to_numeric(today_posts[col], errors='coerce')
    today_posts['created_local'] = pd.to_datetime(
        today_posts.get('created_local'), errors='coerce')

platform_counts = {}
if today_count > 0 and 'platform' in today_posts.columns:
    platform_counts = today_posts['platform'].value_counts().to_dict()

melb_total = summary['by_city'].get('melbourne', 0)

print('=' * 60)
print('   REAL-TIME SYSTEM STATUS DASHBOARD')
print('=' * 60)
print(f'   ES cluster status         : {health.get("status", "unknown")}')
print(f'   ES version                : {health.get("es_version", "unknown")}')
print(f'   Total Melbourne posts     : {melb_total:,}')
print(f'   Full corpus date range    : {summary["date_range"][0]} → {summary["date_range"][1]}')
print(f'   Posts sampled today       : {today_count}')
print(f'   Platform breakdown today  : {platform_counts}')
print(f'   Recent post sample        : {"AVAILABLE" if today_count > 0 else "EMPTY"}')
print('=' * 60)

---

## 6.2 Harvester Health Analysis

Check harvesting continuity over the past 7 days — are there gaps?

In [ ]:
# ── Daily volume from server-side aggregation ───────────────
# Use mode=daily to get total post counts per day (full coverage)
daily_7d_payload = api_get(mode="daily", city="melbourne",
                           **{"from": SEVEN_DAYS_AGO, "to": TODAY})
daily_7d = pd.DataFrame(daily_7d_payload.get("days", []))
require_frame(daily_7d, ["date", "count", "sentiment_mean", "tmax_mean", "prcp_mean"], "Melbourne recent daily aggregates")

if len(daily_7d) > 0:
    daily_7d['date'] = pd.to_datetime(daily_7d['date'])
    daily_7d.rename(columns={'count': 'n_posts', 'sentiment_mean': 'sent_mean',
                              'tmax_mean': 'tmax', 'prcp_mean': 'prcp'}, inplace=True)

# ── Platform breakdown per day (sample from posts) ──────────
# Fetch a sample for each of the last 7 days to show platform mix
platform_daily_rows = []
for i in range(8):
    d = (datetime.now(ZoneInfo("Australia/Melbourne")) - timedelta(days=7-i)).strftime('%Y-%m-%d')
    p = api_get(mode="posts", city="melbourne", size=500,
                **{"from": d, "to": d})
    rows = p.get("rows", [])
    if rows:
        df = pd.DataFrame(rows)
        if 'platform' in df.columns:
            for plat, cnt in df['platform'].value_counts().items():
                platform_daily_rows.append({'date': d, 'platform': plat, 'count': cnt})

platform_daily = pd.DataFrame(platform_daily_rows)

# ── Plot ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left: total daily volume
axes[0].bar(daily_7d['date'].dt.strftime('%m-%d'), daily_7d['n_posts'],
           color=CITY_COLORS['melbourne'], alpha=0.85,
           edgecolor='white', linewidth=1.0)
for i, row in daily_7d.iterrows():
    axes[0].text(i, row['n_posts'] + daily_7d['n_posts'].max() * 0.02,
                str(int(row['n_posts'])),
                ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Daily Harvest Volume (Past 7 Days)')
axes[0].set_ylabel('Posts collected')
axes[0].tick_params(axis='x', rotation=30)
axes[0].margins(y=0.15)

# Right: stacked platform breakdown
if len(platform_daily) > 0:
    pivot = (platform_daily.pivot_table(index='date', columns='platform',
                                         values='count', fill_value=0))
    # Keep platforms in consistent order
    plat_order = [p for p in ['reddit', 'mastodon', 'bluesky'] if p in pivot.columns]
    pivot = pivot[plat_order]
    pivot.plot(kind='bar', stacked=True, ax=axes[1],
              color=[PLATFORM_COLORS[c] for c in pivot.columns],
              alpha=0.9, edgecolor='white', linewidth=0.5, width=0.7)
    axes[1].set_title('Platform Breakdown (Past 7 Days, sample)')
    axes[1].set_ylabel('Posts (sample)')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].legend(title='Platform', loc='upper right')

plt.tight_layout()
plt.show()

total_7d = daily_7d['n_posts'].sum()
days_with_data = len(daily_7d[daily_7d['n_posts'] > 0])
print(f'Total posts in past 7 days  : {total_7d:,}')
print(f'Days with data              : {days_with_data} / {len(daily_7d)}')
print(f'Coverage                    : {days_with_data/len(daily_7d)*100:.0f}%')

---

## 6.3 Today's Melbourne Weather & Mood

A snapshot of today: current weather conditions paired with aggregate social media sentiment.

In [ ]:
# ── Today's weather from daily aggregation ──────────────────
today_daily_payload = api_get(mode="daily", city="melbourne",
                               **{"from": TODAY, "to": TODAY})
today_weather = today_daily_payload.get("days", [])

if today_weather:
    w = today_weather[0]
    temp_now = round(w['tmax_mean'], 1) if pd.notna(w.get('tmax_mean')) else 'N/A'
    rain = round(w['prcp_mean'], 1) if pd.notna(w.get('prcp_mean')) else 'N/A'
else:
    temp_now = rain = 'N/A'

# ── Today's sentiment ───────────────────────────────────────
if today_count > 0:
    avg_sent = today_posts['sentiment'].mean()
    pos_count = (today_posts['sentiment'] > 0.05).sum()
    neg_count = (today_posts['sentiment'] < -0.05).sum()
    neu_count = today_count - pos_count - neg_count
    pos_pct = pos_count / today_count * 100
    neg_pct = neg_count / today_count * 100
else:
    avg_sent = float('nan')
    pos_pct = neg_pct = 0
    pos_count = neg_count = neu_count = 0

avg_sent_label = f'{avg_sent:+.3f}' if today_count else 'N/A'

# ── Historical baseline from full Melbourne daily data ──────
melb_daily_full = api_get(mode="daily", city="melbourne")
melb_daily_df = pd.DataFrame(melb_daily_full.get("days", []))
require_frame(melb_daily_df, ["count", "sentiment_mean", "tmax_mean"], "Melbourne historical daily aggregates")
melb_daily_df.rename(columns={'count': 'n_posts',
                               'sentiment_mean': 'sent_mean'}, inplace=True)
melb_daily_df = melb_daily_df.dropna(subset=['sent_mean'])
if melb_daily_df['n_posts'].sum() <= 0:
    raise RuntimeError('Melbourne historical sentiment requires nonzero post counts.')
hist_melb_sent = (melb_daily_df['sent_mean'] * melb_daily_df['n_posts']).sum() / melb_daily_df['n_posts'].sum()

print('=' * 60)
print(f'   TODAY: {TODAY}')
print('=' * 60)
print('   WEATHER')
print(f'   Max Temperature   : {temp_now}°C')
print(f'   Rainfall          : {rain} mm')
print()
print('   SOCIAL MEDIA MOOD')
print(f'   Posts today        : {today_count}')
print(f'   Avg sentiment      : {avg_sent_label}  (Melbourne historical avg: {hist_melb_sent:+.3f})')
print(f'   Positive posts     : {pos_count} ({pos_pct:.0f}%)')
print(f'   Neutral posts      : {neu_count}')
print(f'   Negative posts     : {neg_count} ({neg_pct:.0f}%)')
print('=' * 60)

In [ ]:
# ── Sentiment distribution today ────────────────────────────
if today_count > 5:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    labels = ['Positive', 'Neutral', 'Negative']
    sizes = [pos_count, neu_count, neg_count]
    colors = [SENTIMENT_COLORS['positive'],
              SENTIMENT_COLORS['neutral'],
              SENTIMENT_COLORS['negative']]
    axes[0].pie(sizes, labels=labels, colors=colors,
               autopct='%1.0f%%', startangle=90,
               wedgeprops={'edgecolor': 'white', 'linewidth': 2},
               textprops={'fontsize': 11})
    axes[0].set_title(f"Today's Sentiment Split ({today_count} posts)")

    axes[1].hist(today_posts['sentiment'].dropna(), bins=30,
                color=CITY_COLORS['melbourne'], alpha=0.75,
                edgecolor='white', linewidth=0.5)
    axes[1].axvline(avg_sent, color='#D32F2F', linewidth=2, linestyle='--',
                    label=f'Today mean: {avg_sent:+.3f}')
    axes[1].axvline(hist_melb_sent, color='#666666', linewidth=1.5, linestyle=':',
                    label=f'Historical avg: {hist_melb_sent:+.3f}')
    axes[1].set_title('Sentiment Distribution Today')
    axes[1].set_xlabel('VADER compound score')
    axes[1].set_ylabel('Count')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('Not enough posts today for distribution plot.')

---

## 6.4 Today's Sentiment by Platform

Compare today's sentiment across Reddit, Mastodon and BlueSky.

In [ ]:
if today_count > 0 and 'platform' in today_posts.columns:
    plat_summary = (today_posts.groupby('platform')['sentiment']
                    .agg(['mean', 'count'])
                    .sort_values('count', ascending=False))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    bar_c = [PLATFORM_COLORS.get(p, '#78909C') for p in plat_summary.index]

    # Left: post count
    axes[0].bar(plat_summary.index, plat_summary['count'],
               color=bar_c, alpha=0.9, edgecolor='white', linewidth=1.2)
    for i, (idx, row) in enumerate(plat_summary.iterrows()):
        axes[0].text(i, row['count'] + plat_summary['count'].max() * 0.02,
                    f'{int(row["count"])}',
                    ha='center', fontweight='bold')
    axes[0].set_title("Today's Posts by Platform")
    axes[0].set_ylabel('Number of posts')
    axes[0].margins(y=0.12)

    # Right: sentiment by platform
    axes[1].bar(plat_summary.index, plat_summary['mean'],
               color=bar_c, alpha=0.9, edgecolor='white', linewidth=1.2)
    for i, (idx, row) in enumerate(plat_summary.iterrows()):
        axes[1].text(i, row['mean'] + 0.008,
                    f'{row["mean"]:+.3f}', ha='center',
                    fontweight='bold')
    axes[1].axhline(0, color='#888888', linewidth=0.6, linestyle=':')
    axes[1].axhline(hist_melb_sent, color='#666666', linewidth=1.5, linestyle='--',
                    alpha=0.7, label=f'Historical avg ({hist_melb_sent:+.3f})')
    axes[1].set_title("Today's Sentiment by Platform")
    axes[1].set_ylabel('Mean sentiment')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print(f'{"Platform":12s}  {"Posts":>6s}  {"Sentiment":>10s}')
    print('-' * 36)
    for plat, row in plat_summary.iterrows():
        print(f'{plat:12s}  {int(row["count"]):6d}  {row["mean"]:+10.3f}')
else:
    print('No posts today or no platform data.')

---

## 6.5 Hourly Sentiment Trend — Today

Does sentiment shift as the day progresses?

In [ ]:
if today_count > 5 and 'created_local' in today_posts.columns and today_posts['created_local'].notna().any():
    today_posts['hour'] = today_posts['created_local'].dt.hour

    hourly = (today_posts.groupby('hour')
              .agg(avg_sentiment=('sentiment', 'mean'),
                   post_count=('sentiment', 'count'))
              .reset_index())

    fig, ax1 = plt.subplots(figsize=(13, 4.5))

    ax_bg = ax1.twinx()
    ax_bg.bar(hourly['hour'], hourly['post_count'], alpha=0.12,
             color='#888888', width=0.8)
    ax_bg.set_ylabel('')
    ax_bg.set_yticks([])

    ax1.plot(hourly['hour'], hourly['avg_sentiment'],
            marker='o', color=CITY_COLORS['melbourne'],
            linewidth=2.5, markersize=8, label='Avg sentiment', zorder=5)
    ax1.fill_between(hourly['hour'], hourly['avg_sentiment'],
                    alpha=0.1, color=CITY_COLORS['melbourne'])
    ax1.set_ylabel('Sentiment (VADER compound)', color=CITY_COLORS['melbourne'])
    ax1.set_xlabel('Hour of day')
    ax1.axhline(0, color='#888888', linewidth=0.6, linestyle=':')
    ax1.axhline(hist_melb_sent, color='#666666', linewidth=1.5, linestyle='--',
               alpha=0.7, label=f'Historical avg ({hist_melb_sent:+.3f})')
    ax1.set_xticks(range(0, 24))
    ax1.set_title(f'Hourly Sentiment Trend — {TODAY}')
    ax1.legend(loc='upper left')

    plt.tight_layout()
    plt.show()

    peak_hour = hourly.loc[hourly['post_count'].idxmax(), 'hour']
    peak_count = hourly['post_count'].max()
    print(f'Peak hour: {int(peak_hour):02d}:00 ({int(peak_count)} posts)')
else:
    print('Not enough posts today for hourly analysis.')

---

## 6.6 Weather–Sentiment Correlation (Past 7 Days)

Is there a measurable relationship between daily temperature and sentiment in the live data?

In [ ]:
if len(daily_7d) >= 3:
    daily_7d_clean = daily_7d.dropna(subset=['sent_mean', 'tmax'])

    if len(daily_7d_clean) >= 3 and daily_7d_clean['tmax'].nunique() > 1:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

        # Left: dual-axis time series
        ax1 = axes[0]
        ax1.plot(daily_7d_clean['date'], daily_7d_clean['sent_mean'],
                marker='o', color=CITY_COLORS['melbourne'],
                linewidth=2.2, markersize=7, label='Sentiment')
        ax1.set_ylabel('Sentiment', color=CITY_COLORS['melbourne'])
        ax1.axhline(0, color='#888888', linewidth=0.6, linestyle=':')
        ax1.tick_params(axis='x', rotation=40)
        ax_t = ax1.twinx()
        ax_t.plot(daily_7d_clean['date'], daily_7d_clean['tmax'],
                 marker='x', color='#C62828', linewidth=2,
                 markersize=8, linestyle='--', label='Temp °C')
        ax_t.set_ylabel('Temperature (°C)', color='#C62828')
        ax1.set_title('Daily Trend (Past 7 Days)')

        # Right: scatter
        axes[1].scatter(daily_7d_clean['tmax'], daily_7d_clean['sent_mean'],
                       c=daily_7d_clean['tmax'], cmap='RdYlBu_r',
                       s=daily_7d_clean['n_posts'] * 0.5, alpha=0.85,
                       edgecolors='black', linewidth=0.6)
        if len(daily_7d_clean) >= 3 and daily_7d_clean['tmax'].nunique() > 1:
            pr, pp = pearsonr(daily_7d_clean['tmax'], daily_7d_clean['sent_mean'])
            z = np.polyfit(daily_7d_clean['tmax'], daily_7d_clean['sent_mean'], 1)
            p_line = np.poly1d(z)
            x_range = np.linspace(daily_7d_clean['tmax'].min(),
                                  daily_7d_clean['tmax'].max(), 50)
            axes[1].plot(x_range, p_line(x_range),
                        color='#C62828', linestyle='--', linewidth=2, alpha=0.7)
            axes[1].set_title(f'Temp vs Sentiment (r={pr:+.3f}, p={pp:.3f})')

        axes[1].set_xlabel('Max temperature (°C)')
        axes[1].set_ylabel('Mean sentiment')
        axes[1].axhline(0, color='#888888', linewidth=0.6, linestyle=':')

        plt.tight_layout()
        plt.show()

        print(f'7-day correlation: r={pr:+.3f}, p={pp:.3f}')
        print(f'(Note: n={len(daily_7d_clean)} days — too few for statistical significance)')
        print('Original project snapshot: Melbourne r=+0.042, p=0.037')
else:
    print('Not enough daily data for correlation analysis.')

---

## 6.7 Rain Effect (Past 7 Days)

Does rain affect Melbourne's social media mood?

In [ ]:
# Use prcp >= 1mm to filter out drizzle / trace rain
RAIN_THRESHOLD = 1.0

if len(daily_7d) >= 3 and 'prcp' in daily_7d.columns:
    daily_7d['is_rainy'] = daily_7d['prcp'] >= RAIN_THRESHOLD

    if daily_7d['is_rainy'].nunique() == 2:
        rainy = daily_7d[daily_7d['is_rainy']]
        dry = daily_7d[~daily_7d['is_rainy']]

        fig, ax = plt.subplots(figsize=(9, 4.5))
        bars = ax.bar(['Dry Days\n(<1mm)', 'Rainy Days\n(≥1mm)'],
                      [dry['sent_mean'].mean(), rainy['sent_mean'].mean()],
                      color=['#FF8F00', '#1F6FB4'], alpha=0.9,
                      edgecolor='white', linewidth=1.2, width=0.55)
        for bar, val, n in zip(bars,
                               [dry['sent_mean'].mean(), rainy['sent_mean'].mean()],
                               [len(dry), len(rainy)]):
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.008,
                   f'{val:+.3f}\n(n={n} days)',
                   ha='center', fontweight='bold')
        ax.axhline(0, color='#888888', linewidth=0.6, linestyle=':')
        ax.axhline(hist_melb_sent, color='#666666', linewidth=1.5, linestyle='--',
                   alpha=0.7, label=f'Historical avg ({hist_melb_sent:+.3f})')
        ax.set_title(f'Real-time: Rainy vs Dry Days (Past 7 Days, threshold ≥{RAIN_THRESHOLD}mm)')
        ax.set_ylabel('Mean sentiment')
        ax.legend()
        ax.margins(y=0.2)
        plt.tight_layout()
        plt.show()

        print(f'Dry days   : {len(dry)} days, mean sentiment = {dry["sent_mean"].mean():+.3f}')
        print(f'Rainy days : {len(rainy)} days, mean sentiment = {rainy["sent_mean"].mean():+.3f}')
        print(f'(Threshold: prcp ≥ {RAIN_THRESHOLD}mm)')
    else:
        weather_type = 'all rainy' if daily_7d['is_rainy'].all() else 'all dry'
        print(f'Past 7 days were {weather_type} (using ≥{RAIN_THRESHOLD}mm threshold) — cannot compare.')
        print('\nDaily rainfall observed:')
        for _, row in daily_7d.iterrows():
            print(f'  {row["date"].strftime("%Y-%m-%d")}: {row["prcp"]:.2f}mm')
else:
    print('Not enough data for rain effect analysis.')

---

## 6.8 What Are Melburnians Talking About?

Word cloud of today's posts.

In [ ]:
stop = (set(STOPWORDS) |
        {'melbourne', 'melb', 'https', 'http', 'com', 'amp',
         'www', 'just', 'like', 'know', 'got', 'one', 'get',
         'really', 'would', 'also', 'much', 'going', 'still',
         'even', 'well', 'back', 'right', 'think', 'good',
         'time', 'people', 'make', 'way', 'want', 'need',
         'removed', 'deleted', 'nan'})

if today_count > 10 and 'text' in today_posts.columns:
    all_text = ' '.join(today_posts['text'].dropna().astype(str).tolist())

    if not WordCloud(stopwords=stop).process_text(all_text):
        raise RuntimeError('The returned posts contain no usable terms for a word cloud.')
    wc = WordCloud(width=1000, height=420, stopwords=stop,
                   background_color='white', collocations=False,
                   colormap='viridis', max_words=80,
                   relative_scaling=0.5).generate(all_text)

    plt.figure(figsize=(13, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    weather_label = f'{temp_now}°C' if temp_now != 'N/A' else 'weather data pending'
    plt.title(f'What Melburnians Are Posting Today — {TODAY} ({weather_label})')
    plt.tight_layout()
    plt.show()

    # Weather-related words check
    weather_words = ['hot', 'heat', 'cold', 'rain', 'sunny', 'wind',
                     'storm', 'warm', 'cool', 'weather', 'umbrella',
                     'freezing', 'humid', 'heatwave', 'drizzle', 'fog']
    found = [w for w in weather_words if w in all_text.lower()]
    if found:
        print(f'Weather-related words found: {found}')
    else:
        print('No obvious weather-related vocabulary detected today.')

    print()
    print('Most POSITIVE posts today:')
    print('-' * 60)
    for _, row in today_posts.nlargest(3, 'sentiment').iterrows():
        text = str(row.get('text', ''))[:120]
        plat = row.get('platform', '?')
        print(f'  [{row["sentiment"]:+.3f}] ({plat}) {text}')

    print()
    print('Most NEGATIVE posts today:')
    print('-' * 60)
    for _, row in today_posts.nsmallest(3, 'sentiment').iterrows():
        text = str(row.get('text', ''))[:120]
        plat = row.get('platform', '?')
        print(f'  [{row["sentiment"]:+.3f}] ({plat}) {text}')
else:
    print('Not enough posts today for word cloud.')

---

## 6.9 Today vs. Melbourne Historical Data

Plot today's real-time reading on top of Melbourne's full historical temperature-sentiment scatter. This provides a descriptive comparison with the historical analysis.

In [ ]:
# ── Plot today's point on top of full historical scatter ────
melb_hist = melb_daily_df.copy()
melb_hist['tmax'] = pd.to_numeric(
    pd.DataFrame(melb_daily_full.get("days", []))['tmax_mean'], errors='coerce')
melb_hist = melb_hist.dropna(subset=['tmax', 'sent_mean'])
if len(melb_hist) < 3 or melb_hist['tmax'].nunique() < 2:
    raise RuntimeError('Historical correlation needs at least three valid daily samples and varying temperatures.')

fig, ax = plt.subplots(figsize=(11, 5.5))

# Historical scatter
ax.scatter(melb_hist['tmax'], melb_hist['sent_mean'],
          alpha=0.22, color=CITY_COLORS['melbourne'], s=28,
          label=f'Melbourne historical ({len(melb_hist):,} days)')

# Regression line
z = np.polyfit(melb_hist['tmax'], melb_hist['sent_mean'], 1)
p_line = np.poly1d(z)
x_range = np.linspace(melb_hist['tmax'].min(), melb_hist['tmax'].max(), 50)
ax.plot(x_range, p_line(x_range), '--',
        color=CITY_COLORS['melbourne'], alpha=0.6, linewidth=2,
        label='Historical trend')

# Today's point
if temp_now != 'N/A' and today_count > 0:
    ax.scatter(float(temp_now), avg_sent,
              s=350, color='#D32F2F', zorder=10,
              edgecolors='black', linewidth=1.5,
              label=f'TODAY ({temp_now}°C, sent={avg_sent:+.3f})')
    ax.annotate('TODAY',
               xy=(float(temp_now), avg_sent),
               xytext=(float(temp_now) + 2.5, avg_sent + 0.06),
               fontsize=12, fontweight='bold', color='#D32F2F',
               arrowprops=dict(arrowstyle='->', color='#D32F2F', lw=1.5))

    # Compare to historical at same temperature (±2°C)
    temp_range = melb_hist[
        (melb_hist['tmax'] >= float(temp_now) - 2) &
        (melb_hist['tmax'] <= float(temp_now) + 2)
    ]
    if len(temp_range) > 0:
        hist_avg_at_temp = temp_range['sent_mean'].mean()
        diff = avg_sent - hist_avg_at_temp
        direction = 'above' if diff > 0 else 'below'
        print(f"Today's sentiment              : {avg_sent:+.3f}")
        print(f'Historical avg at ~{temp_now}°C : {hist_avg_at_temp:+.3f}')
        print(f'Difference                     : {diff:+.3f} ({direction} historical)')

ax.set_xlabel('Max temperature (°C)')
ax.set_ylabel('Mean daily sentiment')
ax.set_title("Today's Melbourne vs Historical Temperature–Sentiment")
ax.axhline(0, color='#888888', linewidth=0.6, linestyle=':')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

pr, pp = pearsonr(melb_hist['tmax'], melb_hist['sent_mean'])
print(f'\nMelbourne historical temp-sentiment correlation: r={pr:+.3f}, p={pp:.3f}')
print('Compare this computed correlation with a separately recorded historical analysis.')

---

## 6.10 Summary & Key Findings

Automated summary of today's real-time analysis.

In [ ]:
print('=' * 60)
print('   REAL-TIME MONITORING SUMMARY')
print('=' * 60)
print()
print(f'Date                : {TODAY}')
print(f'Melbourne Weather   : {temp_now}°C, Rainfall: {rain} mm')
print(f'Posts Analysed      : {today_count}')
print(f'Average Sentiment   : {avg_sent_label}  (historical: {hist_melb_sent:+.3f})')
print()

print('Key Observations:')
print(f'  1. Today post sample is {"available" if today_count > 0 else "empty"}')
print()

if temp_now != 'N/A':
    temp_val = float(temp_now)
    if temp_val >= 35:
        print(f'  2. Extreme heat today ({temp_val}°C)')
        print('     Exploratory finding: weak negative effect at extreme heat')
    elif temp_val >= 30:
        print(f'  2. Hot day today ({temp_val}°C)')
    elif temp_val <= 18:
        print(f'  2. Cool day today ({temp_val}°C)')
        print('     Original project snapshot reported Melbourne r=+0.042.')
    else:
        print(f'  2. Mild temperature today ({temp_val}°C)')
        print('     Weather alone does not establish the cause of observed sentiment.')
print()

if rain == 'N/A':
    print('  3. Rainfall data is not available yet.')
elif float(rain) > 0:
    print(f'  3. Rainy day today ({rain} mm)')
else:
    print('  3. No rainfall recorded in the current daily aggregate.')
print()

if today_count == 0:
    print('  4. No current posts available for a sentiment estimate.')
elif avg_sent > 0.1:
    print(f'  4. Overall mood is POSITIVE today ({avg_sent:+.3f})')
elif avg_sent < -0.05:
    print(f'  4. Overall mood is NEGATIVE today ({avg_sent:+.3f})')
else:
    print(f'  4. Overall mood is NEUTRAL today ({avg_sent:+.3f})')

diff_from_hist = avg_sent - hist_melb_sent
if today_count == 0:
    print('     Historical comparison is unavailable without a current sample.')
elif abs(diff_from_hist) > 0.05:
    direction = 'more positive' if diff_from_hist > 0 else 'more negative'
    print(f'     This is {abs(diff_from_hist):.3f} points {direction} than the historical average')
else:
    print('     This is close to the historical average')

print()
print('=' * 60)
print('  Current observations are descriptive, not a causal validation.')
print('  Compare the computed values with a separately recorded historical run.')
print('=' * 60)

---

*Real-time Melbourne Mood Monitor — COMP90024 Assignment 2*  
*Data pipeline: Jupyter Notebook → Fission REST API → ElasticSearch (NeCTAR MRC)*